# MorningNote AI — Demo Notebook

**Course:** Generative AI for Finance  
**Project:** MorningNote AI: An Agentic Financial Briefing Generator for Equity Watchlists

This notebook runs the full workflow from sector selection to model comparison.
Run all cells from top to bottom.

---

## Contents
1. [Setup & Imports](#section-1)
2. [Choose Your Watchlist](#section-2)
3. [Build Watchlist](#section-3)
4. [Fetch Market Data](#section-4)
5. [Load Company News](#section-5)
6. [Build Prompt](#section-6)
7. [Generate Morning Notes (Two Models)](#section-7)
8. [Baseline — No AI](#section-8)
9. [Compare Model Outputs](#section-9)
10. [Fact-Check Claims](#section-10)
11. [Conclusion and Limitations](#section-11)

---
<a id='section-1'></a>
## Section 1: Setup & Imports

In [ ]:
import os
import sys
import pandas as pd
from datetime import date
from dotenv import load_dotenv
from IPython.display import display, Markdown

sys.path.insert(0, os.path.abspath('.'))
load_dotenv()

from src.sector_mapping import (
    get_tickers_for_sector, list_available_sectors,
    get_sector_description, build_ticker_list,
)
from src.market_data import (
    get_market_data, format_market_data_for_prompt,
    get_market_snapshot, format_snapshot_for_prompt,
    get_earnings_calendar, format_earnings_for_prompt,
)
from src.news_loader import load_news_for_tickers
from src.prompt_builder import build_morningnote_prompt
from src.model_runner import run_openai_model, run_anthropic_model, save_note
from src.evaluator import create_model_comparison_template, create_fact_check_template, print_evaluation_instructions

print('✓ All imports successful.')
print(f'  OpenAI key set:    {bool(os.getenv("OPENAI_API_KEY"))}')
print(f'  Anthropic key set: {bool(os.getenv("ANTHROPIC_API_KEY"))}')
print(f'  NewsAPI key set:   {bool(os.getenv("NEWSAPI_KEY"))}')

---
<a id='section-2'></a>
## Section 2: Choose Your Watchlist

**Three modes available — pick one by setting `MODE` below:**

| Mode | What it does |
|------|--------------|
| `"sector"` | Pick a pre-built sector watchlist (e.g. AI & Semiconductors) |
| `"custom"` | Enter your own ticker symbols (e.g. TSLA, SPOT, COIN) |
| `"mixed"` | Start from a sector, then add extra tickers on top |

In [ ]:
# ── STEP 1: Pick a mode ──────────────────────────────────────────────────────
# Options: "sector" | "custom" | "mixed"
MODE = 'sector'

# ── STEP 2: Configure based on your mode ────────────────────────────────────

# Used when MODE = "sector" or "mixed"
# Run the next cell to see all available sectors
SELECTED_SECTOR = 'AI & Semiconductors'

# Used when MODE = "custom" or "mixed"
# Enter any valid US stock tickers separated by commas
CUSTOM_TICKERS = 'TSLA, COIN, RBLX'

# ─────────────────────────────────────────────────────────────────────────────

print('Available sectors:')
for s in list_available_sectors():
    print(f'  • {s}: {get_sector_description(s)}')

print(f'\nSelected mode:   {MODE}')
if MODE in ('sector', 'mixed'):
    print(f'Selected sector: {SELECTED_SECTOR}')
if MODE in ('custom', 'mixed'):
    print(f'Custom tickers:  {CUSTOM_TICKERS}')

---
<a id='section-3'></a>
## Section 3: Build Watchlist

Converts your mode selection into a final ticker list.

In [ ]:
tickers, note_label = build_ticker_list(
    mode         = MODE,
    sector_name  = SELECTED_SECTOR if MODE in ('sector', 'mixed') else None,
    custom_input = CUSTOM_TICKERS  if MODE in ('custom', 'mixed') else None,
)

print(f'Mode:        {MODE}')
print(f'Note label:  {note_label}')
print(f'Tickers:     {tickers}')
print(f'Count:       {len(tickers)} companies')

---
<a id='section-4'></a>
## Section 4: Fetch Market Data

Three types of real data from yfinance — all free, no API key needed:
- **Broad market snapshot**: major indices, VIX, commodities, FX, bonds
- **Stock prices**: the tickers in your watchlist
- **Earnings calendar**: next scheduled earnings date per ticker

In [ ]:
print('Fetching broad market snapshot...')
snapshot_df = get_market_snapshot()
print(f'✓ {len(snapshot_df)} instruments\n')
display(snapshot_df)

In [ ]:
print('Fetching stock prices from yfinance...')
market_df = get_market_data(tickers)
print(f'✓ {len(market_df)} tickers\n')
display(market_df)

In [ ]:
print('Fetching earnings calendar...')
earnings_df = get_earnings_calendar(tickers)
print()
display(earnings_df)

---
<a id='section-5'></a>
## Section 5: Load Company News

Fetches real headlines via NewsAPI (last 3 days).
Automatically falls back to `data/sample_news.csv` if the API is unavailable.

In [ ]:
print('Loading news headlines...')
news_df = load_news_for_tickers(tickers)
print(f'\n✓ {len(news_df)} articles loaded.\n')
display(news_df)

---
<a id='section-6'></a>
## Section 6: Build Prompt

Combines all four data sources into one structured prompt with anti-hallucination instructions.

In [ ]:
prompt = build_morningnote_prompt(
    sector      = note_label,
    tickers     = tickers,
    market_df   = market_df,
    news_df     = news_df,
    snapshot_df = snapshot_df,
    earnings_df = earnings_df,
)

print(f'✓ Prompt built: {len(prompt)} characters.\n')
print('--- Prompt Preview (first 1000 chars) ---')
print(prompt[:1000])
print('...')

---
<a id='section-7'></a>
## Section 7: Generate Morning Notes (Two Models)

Both models receive the **exact same prompt** — fair comparison.

- **Model A:** OpenAI `gpt-4o-mini`
- **Model B:** Anthropic `claude-haiku-4-5-20251001`

In [ ]:
print('Generating Model A (OpenAI gpt-4o-mini)...')
model_a_output = run_openai_model(prompt, model_name='gpt-4o-mini')
save_note(model_a_output, 'outputs/model_a_morning_note.md')
print('✓ Model A done.\n')

print('Generating Model B (Anthropic claude-haiku-4-5-20251001)...')
model_b_output = run_anthropic_model(prompt, model_name='claude-haiku-4-5-20251001')
save_note(model_b_output, 'outputs/model_b_morning_note.md')
print('✓ Model B done.')

In [ ]:
print('=' * 60)
print('MODEL A — OpenAI gpt-4o-mini')
print('=' * 60)
display(Markdown(model_a_output))

print('\n')
print('=' * 60)
print('MODEL B — Anthropic Claude Haiku')
print('=' * 60)
display(Markdown(model_b_output))

---
<a id='section-8'></a>
## Section 8: Baseline — No AI

Raw data with no summarization. Comparing this to the AI notes shows whether AI adds value.

In [ ]:
headlines = '\n'.join(
    f"  - [{row['date']}] {row['ticker']}: {row['headline']} ({row['source']})"
    for _, row in news_df.iterrows()
)

baseline = f"""# Baseline Morning Note (No AI)

Date:      {date.today()}
Watchlist: {note_label}
Tickers:   {', '.join(tickers)}

## Broad Market Snapshot

{format_snapshot_for_prompt(snapshot_df)}

## Stock Prices

{market_df.to_string(index=False)}

## Earnings Calendar

{format_earnings_for_prompt(earnings_df)}

## Raw Headlines

{headlines}
"""

with open('outputs/baseline_note.md', 'w', encoding='utf-8') as f:
    f.write(baseline)

print('✓ Baseline saved to outputs/baseline_note.md\n')
print(baseline)

---
<a id='section-9'></a>
## Section 9: Compare Model Outputs

Generate the blank scoring table — **fill it in manually** after reading both outputs.

In [ ]:
comparison_df = create_model_comparison_template(
    model_a_name='GPT-4o-mini',
    model_b_name='Claude Haiku',
)
print()
display(comparison_df[['dimension', 'description', 'GPT-4o-mini', 'Claude Haiku', 'notes']])

# After filling in the CSV, uncomment to reload:
# display(pd.read_csv('outputs/model_comparison.csv'))

---
<a id='section-10'></a>
## Section 10: Fact-Check Claims

Generate the blank fact-check table — **fill it in manually** after reviewing both outputs.

For each claim: **Yes** (supported) / **Partial** / **No** (hallucination).

In [ ]:
fact_check_df = create_fact_check_template()
print()
display(fact_check_df)

# After filling in the CSV, uncomment to reload:
# display(pd.read_csv('outputs/fact_check_table.csv'))

In [ ]:
print_evaluation_instructions()

---
<a id='section-11'></a>
## Section 11: Conclusion and Limitations

**What worked well:**
> Fill in after evaluation.

**Which model performed better and why:**
> Fill in after scoring — refer to model_comparison.csv.

**Failure case:**
> Describe one example where a model made an unsupported causal claim.

**Limitations:**
1. Sector-to-ticker mapping is manually curated, not dynamically updated.
2. NewsAPI free tier limits history to 1 month and 100 requests/day.
3. LLMs may overstate causality even with anti-hallucination instructions.
4. Earnings dates from yfinance may be estimates, not confirmed dates.
5. Human review is still required before acting on AI-generated financial content.

---
*This notebook is not investment advice. All outputs require human review.*